In [1]:
from Lab_Equipment.Config import config 

Current Directory: c:\Users\Experiment\Documents\RelBohmTraj\ExperimentStuff
This is a Windows system.
Deformable Mirror Software not installed. If needed to go and get software from USB website.


In [ ]:
# Python Libs
import cv2
import numpy as np
import matplotlib.pyplot as plt
import copy
from IPython.display import display, clear_output
import ipywidgets
import time
import scipy.io
import TimeTagger
import os
import threading
import datetime

from scipy import io, integrate, linalg, signal
from scipy.io import savemat, loadmat
from scipy.fft import fft, fftfreq, fftshift,ifftshift, fft2,ifft2,rfft2,irfft2
# Defult ploting properties 
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = [5,5]




# timeTagger
import Lab_Equipment.TimeTagger.TimeTaggerLiveWindow as TimeTaggerLiveWindow
import Lab_Equipment.TimeTagger.TimeTaggerFunction as TimetaggerFunc
import Lab_Equipment.TimeTagger.TimeTaggerCustomMeasurementObj as TimeTaggerCustomObj



# Piezo translation stage
import Lab_Equipment.ThorlabsKCube.ThorlabsPiezoStrainGauge as Piezolib

#Power Meter
import Lab_Equipment.PowerMeter.PowerMeter_Thorlabs_lib as PwrMeterLib


## Power Meter

In [ ]:
del PwrMeterObj1
del PwrMeterObj2

In [ ]:
PwrMeterObj1=PwrMeterLib.PowerMeterObj(deviceName='USB0::0x1313::0x8078::P0024994::INSTR',AvgCount=50)
PwrMeterObj2=PwrMeterLib.PowerMeterObj(deviceName='USB0::0x1313::0x8078::P0035211::INSTR',AvgCount=50)

PwrMeterObj1.SetAverageMeasure(100)
PwrMeterObj1.GetPower()

PwrMeterObj2.SetAverageMeasure(100)
PwrMeterObj2.GetPower()


In [ ]:
PwrMeterObj1.SetAverageMeasure(10000)
PwrMeterObj2.SetAverageMeasure(10000)


#print(PwrMeterObj1.GetPower(),PwrMeterObj2.GetPower())


print(PwrMeterObj2.GetPower())

In [ ]:
num_measurements = 10

# Storage for power readings
power_readings_1 = np.zeros(num_measurements)
power_readings_2 = np.zeros(num_measurements)

# Measurement function using threading
def get_power_pair():
    result = [None, None]

    def read1():
        result[0] = PwrMeterObj1.GetPower()

    def read2():
        result[1] = PwrMeterObj2.GetPower()

    t1 = threading.Thread(target=read1)
    t2 = threading.Thread(target=read2)
    t1.start()
    t2.start()
    t1.join()
    t2.join()
    
    return result[0], result[1]

# Repeat measurements
for i in range(num_measurements):
    p1, p2 = get_power_pair()
    power_readings_1[i]=p1
    power_readings_2[i]=p2
    print(f"Measurement {i+1}: P1 = {p1:.4e}, P2 = {p2:.4e}")



# Compute mean and standard deviation
mean1, std1 = np.mean(power_readings_1), np.std(power_readings_1)
mean2, std2 = np.mean(power_readings_2), np.std(power_readings_2)

print("\n=== Results Summary ===")
print(f"Power Meter 1: Mean = {mean1:.4e} W, Std Dev = {std1:.4e} W")
print(f"Power Meter 2: Mean = {mean2:.4e} W, Std Dev = {std2:.4e} W")

## Piezo Stage

In [ ]:
del PiezoMount

In [4]:
PiezoMount=Piezolib.PiezoStrainGaugeObj()

0
1
A conection has been made to  KPC101 Piezo Controller
Setting Zero Point


In [5]:
# print(PiezoMount.CurrentPostion)
# newPos=PiezoMount.SetPosition(10.003)
# print(newPos)
a=PiezoMount.GetPosition()
print(a)

0


In [ ]:
PiezoMount.SetPosition(0.8401)

## The following is code to check fluctuations in the setup measured with power meters
## This code also can be used to perform the experiment with bright coherent source

In [ ]:
CountTime=60
maxdist=4 # this is in micros
mindist=0 # this is in micros
distSpacing_ideal =0.1 #can go as low as 0.01
distCount=int((maxdist-mindist)/distSpacing_ideal)
distArr_ideal=np.linspace(mindist,maxdist,distCount)
distSpacing=distArr_ideal[1]-distArr_ideal[0]
print(distCount,distSpacing_ideal,distSpacing)
# distCount=10

#Set mount to zero
PiezoMount.SetPosition(0.0001)

In [ ]:
PwrMeterObj1.SetAverageMeasure(10000)  # noticed that this number the measurement happens for around 3 mins. Take 20 such merasurements to cover 1 minute
PwrMeterObj2.SetAverageMeasure(10000)

In [ ]:

num_measurements = 20

# Storage for power readings
power_readings_1 = np.zeros(num_measurements)
power_readings_2 = np.zeros(num_measurements)

# Measurement function using threading
def get_power_pair():
    result = [None, None]

    def read1():
        result[0] = PwrMeterObj1.GetPower()

    def read2():
        result[1] = PwrMeterObj2.GetPower()

    t1 = threading.Thread(target=read1)
    t2 = threading.Thread(target=read2)
    t1.start()
    t2.start()
    t1.join()
    t2.join()
    
    return result[0], result[1]


In [ ]:
#save_dir = "PowerMeterData/Zero_Offset/BS_Scan"
save_dir = "Data/CW_Data/NewData_v1/QWPtest/CoherentPowerMeterMeas_v7"
# save_dir = "Data/CW_Data/NewData_v1/TopticaCWfluc"
os.makedirs(save_dir, exist_ok=True)

num_measurements = 20          # readings per piezo position

HWPoffset = 4.0

maxdist=1 # this is in micron
mindist=0 # this is in micron
distSpacing_ideal =0.02 #can go as low as 0.01
distCount=int((maxdist-mindist)/distSpacing_ideal)
distArr_ideal=np.linspace(mindist,maxdist,distCount)
distSpacing=distArr_ideal[1]-distArr_ideal[0]
print(distCount,distSpacing_ideal,distSpacing)

distArr_measured=np.zeros((distCount))
power_readings_1 = np.zeros((distCount, num_measurements), dtype=float)
power_readings_2 = np.zeros((distCount, num_measurements), dtype=float)

mean1_arr = np.zeros(distCount)
std1_arr  = np.zeros(distCount)
mean2_arr = np.zeros(distCount)
std2_arr  = np.zeros(distCount)

PwrMeterObj1.SetAverageMeasure(10000)
PwrMeterObj2.SetAverageMeasure(10000)

#Set mount to zero
PiezoMount.SetPosition(0.0001)
time.sleep(2.0)

start_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
#outfile = os.path.join(save_dir, f"scan_freespace_{start_ts}.npz")
outfile = os.path.join(save_dir, f"BrightCW_hwp_4_50_50_0_1_0.02_60_fiber_take5.npz")
#outfile = os.path.join(save_dir, f"BrightCW_fluctest_take1.npz")




for idist in range(distCount):
    
    #################################
    #take powermeter measurments
    #################################
    for k in range(num_measurements):
        p1, p2 = get_power_pair()
        power_readings_1[idist, k] = p1
        power_readings_2[idist, k] = p2
        #print(f"    m{k+1:02d}: P1={p1:.4e} W, P2={p2:.4e} W")
        
    # ---- Compute per-position stats (scalars) & store into arrays ----
    m1 = power_readings_1[idist].mean()
    s1 = power_readings_1[idist].std(ddof=1)
    m2 = power_readings_2[idist].mean()
    s2 = power_readings_2[idist].std(ddof=1)

    mean1_arr[idist] = m1
    std1_arr[idist]  = s1
    mean2_arr[idist] = m2
    std2_arr[idist]  = s2

    # Live print (scalars)
    print(f"Power Meter 1: Mean = {m1:.4e} W, Std Dev = {s1:.4e} W")
    print(f"Power Meter 2: Mean = {m2:.4e} W, Std Dev = {s2:.4e} W")
    
    #################################
    # Move the the BeamSplitter
    #################################
    if (idist==0):
        distArr_measured[idist]=float(str(PiezoMount.GetPosition()))
        print(distArr_measured[idist],distArr_ideal[idist])
    else:
       
        distArr_measured[idist]=float(str(PiezoMount.SetPosition(distArr_ideal[idist])))
        time.sleep(2.0)
        print(distArr_measured[idist],distArr_ideal[idist])
        print(PiezoMount.GetPosition())
    # Save after this position (overwrites same file with accumulated data)
    np.savez_compressed(
        outfile,
        distArr_ideal=distArr_ideal,
        distArr_measured=distArr_measured,
        power_readings_1=power_readings_1,
        power_readings_2=power_readings_2,
        mean1=mean1_arr, std1=std1_arr,
        mean2=mean2_arr, std2=std2_arr,
        num_measurements=num_measurements,
        mindist=mindist,
        maxdist=maxdist,
        distSpacing=distSpacing_ideal,
        AngleOfFirstWaveplate=HWPoffset,
    )
    print(f"Saved checkpoint: {outfile}")

### Measurement of density and current separately. It only involves one measurement in the output port. Threading to get power pair is not required

In [ ]:
#save_dir = "PowerMeterData/Zero_Offset/BS_Scan"
save_dir = "Data/CW_Data/NewData_v1/Separate_Meas/CoherentPowerMeterMeas_v1"
os.makedirs(save_dir, exist_ok=True)

num_measurements = 20          # readings per piezo position

#HWPoffset = 5.0

maxdist=1 # this is in micron
mindist=0 # this is in micron
distSpacing_ideal =0.02 #can go as low as 0.01
distCount=int((maxdist-mindist)/distSpacing_ideal)
distArr_ideal=np.linspace(mindist,maxdist,distCount)
distSpacing=distArr_ideal[1]-distArr_ideal[0]
print(distCount,distSpacing_ideal,distSpacing)

distArr_measured=np.zeros((distCount))
power_readings_1 = np.zeros((distCount, num_measurements), dtype=float)
power_readings_2 = np.zeros((distCount, num_measurements), dtype=float)

mean1_arr = np.zeros(distCount)
std1_arr  = np.zeros(distCount)
mean2_arr = np.zeros(distCount)
std2_arr  = np.zeros(distCount)

PwrMeterObj1.SetAverageMeasure(10000)
PwrMeterObj2.SetAverageMeasure(10000)

#Set mount to zero
PiezoMount.SetPosition(0.0001)
time.sleep(1.0)

start_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
#outfile = os.path.join(save_dir, f"scan_freespace_{start_ts}.npz")
outfile = os.path.join(save_dir, f"BrightCW_current_40_60_0_1_0.02_60_take3_nowp.npz")





for idist in range(distCount):
    
    #################################
    #take powermeter measurments
    #################################
    for k in range(num_measurements):
        p1, p2 = get_power_pair()
        power_readings_1[idist, k] = p1
        power_readings_2[idist, k] = p2
        #print(f"    m{k+1:02d}: P1={p1:.4e} W, P2={p2:.4e} W")
        
    # ---- Compute per-position stats (scalars) & store into arrays ----
    m1 = power_readings_1[idist].mean()
    s1 = power_readings_1[idist].std(ddof=1)
    m2 = power_readings_2[idist].mean()
    s2 = power_readings_2[idist].std(ddof=1)

    mean1_arr[idist] = m1
    std1_arr[idist]  = s1
    mean2_arr[idist] = m2
    std2_arr[idist]  = s2

    # Live print (scalars)
    print(f"Power Meter 1: Mean = {m1:.4e} W, Std Dev = {s1:.4e} W")
    print(f"Power Meter 2: Mean = {m2:.4e} W, Std Dev = {s2:.4e} W")
    
    #################################
    # Move the the BeamSplitter
    #################################
    if (idist==0):
        distArr_measured[idist]=float(str(PiezoMount.GetPosition()))
        print(distArr_measured[idist],distArr_ideal[idist])
    else:
       
        distArr_measured[idist]=float(str(PiezoMount.SetPosition(distArr_ideal[idist])))
        time.sleep(1.0)
        print(distArr_measured[idist],distArr_ideal[idist])
        print(PiezoMount.GetPosition())
    # Save after this position (overwrites same file with accumulated data)
    np.savez_compressed(
        outfile,
        distArr_ideal=distArr_ideal,
        distArr_measured=distArr_measured,
        power_readings_1=power_readings_1,
        power_readings_2=power_readings_2,
        mean1=mean1_arr, std1=std1_arr,
        mean2=mean2_arr, std2=std2_arr,
        num_measurements=num_measurements,
        mindist=mindist,
        maxdist=maxdist,
        distSpacing=distSpacing_ideal,
    )
    print(f"Saved checkpoint: {outfile}")

### reading for a single position of the piezo stage

In [ ]:
PiezoMount.SetPosition(1)

In [ ]:
save_dir = "PowerMeterData/Zero_Offset/FixedPosData/"
os.makedirs(save_dir, exist_ok=True)

PiezoPos = 4

start_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
#outfile = os.path.join(save_dir, f"freespace_PiezoPos_{PiezoPos}_{start_ts}.npz")
outfile = os.path.join(save_dir, f"fibercoupled_PiezoPos_{PiezoPos}_{start_ts}.npz")
num_measurements = 300  

power_readings_1 = np.zeros(num_measurements, dtype=float)
power_readings_2 = np.zeros(num_measurements, dtype=float)

PiezoMount.SetPosition(PiezoPos) # set the position of the stage to take reading


for k in range(num_measurements):
    p1, p2 = get_power_pair()
    power_readings_1[k]=p1
    power_readings_2[k]=p2
    
    # Print every 20 measurements
    if (k+1) % 20 == 0:
        print(f"Measurement {k+1}/{num_measurements}: P1={p1:.4e} W, P2={p2:.4e} W")



# Compute mean and standard deviation
mean1, std1 = np.mean(power_readings_1), np.std(power_readings_1)
mean2, std2 = np.mean(power_readings_2), np.std(power_readings_2)


np.savez_compressed(
        outfile,
        power_readings_1=power_readings_1,
        power_readings_2=power_readings_2,
        mean1=mean1, 
        std1=std1,
        mean2=mean2, 
        std2=std2,
        num_measurements=num_measurements
    )
print(f"Saved : {outfile}")       

## CW mesurement with attenuated light

In [ ]:
createtagger=1
if(createtagger==1):
    tagger = TimeTagger.createTimeTagger()
else:
    TimeTagger.freeTimeTagger(tagger)

In [ ]:
sm = TimeTagger.SynchronizedMeasurements(tagger)
sync_tagger = sm.getTagger()

In [ ]:
# Channels (three detectors)
ch1, ch2, ch3 = 1, 2, 3
channels = (ch1, ch2, ch3)
n_channels = len(channels)

## CW with RAWtags

In [3]:
def _intvector(chs):
    """Convert a Python iterable of ints to a TimeTagger.IntVector()."""
    intvec = TimeTagger.IntVector()
    for ch in chs:
        intvec.push_back(int(ch))
    return intvec

def pick_stream_channels(tagger):
    """
    Choose which channels to stream for CW runs.
    We only care about (1, 2, 3) now.

    Tries to intersect with the tagger's registered channels if available;
    otherwise falls back to [1, 2, 3] (the TimeTagStream will simply ignore
    non-existent channels).
    """
    try:
        cfg = tagger.getConfiguration()
        regs = set(cfg.get("registered channels", []))
    except Exception:
        regs = set([1, 2, 3])

    # Prefer the canonical (1,2,3), but only those that actually exist
    cand = [1, 2, 3]
    chosen = [c for c in cand if c in regs]

    # If none of 1,2,3 are registered (weird), just stream whatever is there
    if not chosen and regs:
        chosen = sorted(regs)

    # Final fallback: still try [1,2,3]
    if not chosen:
        chosen = [1, 2, 3]

    return chosen

import threading
import numpy as np
import time

def _decode_buffer_with_timeout(buf, timeout_s=0.5):
    """
    Try to decode a TimeTagStream buffer into (ts, ch, ty) with a watchdog timeout.
    Returns (ts, ch, ty) or (None, None, None) on timeout/failure.

    ts: int64 array of timestamps in ps
    ch: int32 array of channel IDs
    ty: uint8 array of event types (or None if not present)
    """
    out = {}

    def _worker():
        try:
            # Fast path if available
            if hasattr(buf, "toNumpy"):
                arr = buf.toNumpy()
                ts = arr["time"].astype(np.int64,  copy=False)
                ch = arr["channel"].astype(np.int32, copy=False)
                ty = arr["type"].astype(np.uint8,  copy=False) if "type" in arr.dtype.names else None
                out["ts"], out["ch"], out["ty"] = ts, ch, ty
                return

            # Legacy getters
            ts = None
            for ts_name in ("getTimestampsPs", "getTimestamps", "getTimes"):
                if hasattr(buf, ts_name):
                    ts = getattr(buf, ts_name)()
                    break
            if ts is None:
                return

            ch = buf.getChannels() if hasattr(buf, "getChannels") else None
            ty = buf.getTypes()    if hasattr(buf, "getTypes")    else None

            ts = np.asarray(ts, dtype=np.int64)
            ch = np.asarray(ch, dtype=np.int32) if ch is not None else None
            ty = np.asarray(ty, dtype=np.uint8) if ty is not None else None
            out["ts"], out["ch"], out["ty"] = ts, ch, ty
        except Exception:
            return  # leave out empty => signals failure

    t = threading.Thread(target=_worker, daemon=True)
    t.start()
    t.join(timeout_s)
    if t.is_alive():
        return None, None, None

    return out.get("ts"), out.get("ch"), out.get("ty")

def stream_capture_synchronously(tagger, channels,
                                 status_prefix="[stream]",
                                 per_buffer_timeout_s=0.5,
                                 overall_timeout_s=5.0):
    """
    Start a TimeTagStream on `channels` right now and return a function
    stop_and_drain() that will:

      - stop the stream,
      - drain all buffers with per-buffer decode timeout,
      - abort draining after `overall_timeout_s` to avoid hangs,

    and finally return (timestamps_ps, channel_ids, event_types).

    Usage pattern:

        stop_and_drain = stream_capture_synchronously(tagger_for_stream, channels_to_stream)
        sm.startFor(...)
        sm.waitUntilFinished()
        timestamps_ps, channel_ids, event_types = stop_and_drain()
    """
    intvec = _intvector(channels)
    # 2_000_000 is the buffer size in tags; adjust if needed
    stream = TimeTagger.TimeTagStream(tagger, int(40_000_000), intvec)

    if hasattr(stream, "start"):
        stream.start()

    print(f"{status_prefix} START on {channels}; will drain after SM window")

    def stop_and_drain():
        if hasattr(stream, "stop"):
            stream.stop()

        total = 0
        times_list, chans_list, types_list = [], [], []
        started = time.time()
        skipped = 0

        while True:
            if time.time() - started > overall_timeout_s:
                print(f"{status_prefix} drain timeout after {overall_timeout_s}s "
                      f"(kept={total}, skipped={skipped})")
                break

            buf = stream.getData()  # returns next buffer or None
            if buf is None:
                break

            ts, ch, ty = _decode_buffer_with_timeout(
                buf, timeout_s=per_buffer_timeout_s
            )
            if ts is None or ch is None:
                skipped += 1
                continue

            if ts.size:
                total += ts.size
                times_list.append(ts)
                chans_list.append(ch)
                if ty is not None:
                    types_list.append(ty)

        ts_all = np.concatenate(times_list) if times_list else np.empty(0, np.int64)
        ch_all = np.concatenate(chans_list) if chans_list else np.empty(0, np.int32)
        ty_all = np.concatenate(types_list) if types_list else None

        uniq = np.unique(ch_all) if ch_all.size else []
        print(f"{status_prefix} STOP; drained {total} tags; "
              f"skipped {skipped} buffers; unique chans: {uniq}")

        return ts_all, ch_all, ty_all

    return stop_and_drain

In [ ]:
PiezoMount.SetPosition(1.0)
time.sleep(0.2)
print(PiezoMount.GetPosition())

In [ ]:
# -------------------------
# Angles, channels, timing
# -------------------------
Mount1Angle = 4.0
#Mount2Angle = 0.1   #don't use this one for analysis ...just put anything

# ======================================
# Counting settings
# ======================================
num_measurements = 60          # bins per piezo position
bin_duration_s   = 1.0         # 1 s per bin
CountTime_s      = num_measurements * bin_duration_s
binwidth_ps      = int(bin_duration_s * 1e12)



# Channels for CW counts 
channels     = (1, 2, 3)
ChannelCount = len(channels)

# -------------------------
# TimeTagger + Counters
# -------------------------
# tagger = TimeTagger.createTimeTagger()
# sm     = TimeTagger.SynchronizedMeasurements(tagger)
# sync_tagger = sm.getTagger()

counters = []
for ch in channels:
    ctr = TimeTagger.Counter(
        sync_tagger,
        channels=[ch],
        binwidth=binwidth_ps,
        n_values=num_measurements
    )
    counters.append(ctr)

# -------------------------
# Scan settings
# -------------------------
maxdist = 1.0   # microns
mindist = 0.0   # microns
distSpacing_ideal = 0.02
distCount = int((maxdist - mindist) / distSpacing_ideal)
distArr_ideal = np.linspace(mindist, maxdist, distCount)
distSpacing   = distArr_ideal[1] - distArr_ideal[0]
print(distCount, distSpacing_ideal, distSpacing)

#Set mount to zero
PiezoMount.SetPosition(0.0001)
time.sleep(2.0)
# PiezoMount.SetPosition(0.5)
# time.sleep(1.0)

# Arrays: 1D vs position, per channel
distArr_measured = np.zeros(distCount)


total_counts = np.zeros((ChannelCount,distCount), dtype=np.int64)  # integrated counts
mean_rates   = np.zeros((ChannelCount,distCount), dtype=float)     # cps
std_rates    = np.zeros((ChannelCount,distCount), dtype=float)     # cps (from binning)

# -------------------------
# Save paths
# -------------------------
save_dir = "Data/CW_Data/AttenuateCW/WeakCoherentSNSPDMeas_v5"
os.makedirs(save_dir, exist_ok=True)
# start_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
# outfile  = os.path.join(save_dir, f"scan_counts_cw_{start_ts}.npz")
filename = 'WeakCW_hwp_4_30_70_0_1_0.02_60_take3'
outfile = os.path.join(save_dir, f"{filename}.npz")

# choose channels to stream from THIS device/tagger
tagger_for_stream   = sm.getTagger()
channels_to_stream  = pick_stream_channels(tagger_for_stream)  # will choose [1,2,3]
print("[stream] chosen channels:", channels_to_stream)

# base folder for per-position raw stream dumps (CW)
RAW_BASE = os.path.join(save_dir, filename + "_RawTags")
os.makedirs(RAW_BASE, exist_ok=True)

# -------------------------
# Main scan loop 
# -------------------------
for idist in range(distCount):
    
    
    #################################
    # 1) Move the BeamSplitter & record measured position
    #################################

    if idist == 0:
        distArr_measured[idist] = float(str(PiezoMount.GetPosition()))
    else:
        distArr_measured[idist] = float(str(PiezoMount.SetPosition(distArr_ideal[idist])))
        time.sleep(2.0)
        print("Piezo position (reported):", PiezoMount.GetPosition())

    print(f"\n=== Position {idist+1}/{distCount} ===")
    print("Measured position, ideal:", distArr_measured[idist], distArr_ideal[idist])

    #################################
    # 2) per-position raw-tag folder
    #################################
    pos_dir = os.path.join(RAW_BASE, f"RawtagMountPos{idist}")
    os.makedirs(pos_dir, exist_ok=True)

    #################################
    # 3) start stream BEFORE blocking measurement
    #################################
    stop_and_drain = stream_capture_synchronously(
        tagger_for_stream,
        channels_to_stream,
        status_prefix=f"[stream pos {idist}]"
    )

    #################################
    # 4) blocking CW counts measurement (replaces GetCorrelationAndCountsdata_SynMulti)
    #################################
    print(f"\n=== Position index {idist} ===")
    print(f"Counting for {CountTime_s:.1f} s "
          f"in {num_measurements} x {bin_duration_s:.1f} s bins...")

    sm.clear()
    total_time_ps = int(CountTime_s * 1e12)
    sm.startFor(total_time_ps)
    sm.waitUntilFinished()

    #################################
    # 5) Read counters and fill 1D arrays vs position
    #################################
    for ch, ctr in enumerate(counters):
        data_bins = ctr.getData()[0]  # shape (num_measurements,)

        tot       = int(data_bins.sum())
        mean_rate = tot / CountTime_s
        std_rate  = data_bins.std(ddof=1) / bin_duration_s

        total_counts[ch, idist] = tot
        mean_rates[ch, idist]   = mean_rate
        std_rates[ch, idist]    = std_rate

        print(f"Ch {channels[ch]}: total = {tot:d}, "
              f"mean rate = {mean_rate:.3e} cps, std = {std_rate:.3e} cps")

    #################################
    # 6) stop stream & drain → save raw tags (CW)
    #################################
    timestamps_ps, channel_ids, event_types = stop_and_drain()
    raw_npz = os.path.join(pos_dir, f"stream_tags_pos_{idist}.npz")
    np.savez_compressed(
        raw_npz,
        timestamps_ps=timestamps_ps,
        channel_ids=channel_ids,
        event_types=(event_types if event_types is not None else np.array([], dtype=np.uint8)),
        channels_streamed=np.array(channels_to_stream, dtype=np.int32),
        count_time_s=np.array([CountTime_s], dtype=np.float64)
    )
    print(f"[stream pos {idist}] saved raw tags in {raw_npz}")

    

    #################################
    # 7) rolling save of counts 
    #################################
    np.savez_compressed(
        outfile,
        # scan info
        distArr_ideal=distArr_ideal,
        distArr_measured=distArr_measured,
        mindist=mindist,
        maxdist=maxdist,
        distSpacing=distSpacing_ideal,

        # timing metadata
        channels=np.array(channels, dtype=int),
        num_measurements=num_measurements,
        bin_duration_s=bin_duration_s,
        CountTime_s=CountTime_s,
        binwidth_ps=binwidth_ps,

        # waveplate offset angles
        AngleOfFirstWaveplate=Mount1Angle,
        #AngleOfSecondWaveplate=Mount2Angle,

        # counts vs position (1D over stage, 2D with channels)
        total_counts=total_counts,   # (ChannelCount, distCount)
        mean_rates=mean_rates,       # (ChannelCount, distCount)
        std_rates=std_rates,         # (ChannelCount, distCount)
    )
    print(f"Saved checkpoint counts in {outfile}")

### Checking at a particular piezo stage position

In [ ]:
PiezoMount.SetPosition(0.4001)

In [ ]:
print(PiezoMount.GetPosition())

In [ ]:
# -------------------------
# Settings
# -------------------------
num_measurements = 20
bin_duration_s = 1.0
binwidth_ps = int(bin_duration_s * 1e12)

channels = (1, 2, 3)
tau = 0.50   # detection efficiency correction

# -------------------------
# Counters (1 bin per run)

# -------------------------
counters = []
for ch in channels:
    ctr = TimeTagger.Counter(
        sync_tagger,
        channels=[ch],
        binwidth=binwidth_ps,
        n_values=1
    )
    counters.append(ctr)

# -------------------------
# Storage
# -------------------------
rate_readings_1 = np.zeros(num_measurements)
rate_readings_2 = np.zeros(num_measurements)
rate_readings_3 = np.zeros(num_measurements)

# -------------------------
# Sequential 1-second measurements
# -------------------------
for i in range(num_measurements):
    sm.clear()

    sm.startFor(binwidth_ps)
    sm.waitUntilFinished()

    c1 = counters[0].getData()[0][0]
    c2 = counters[1].getData()[0][0]
    c3 = counters[2].getData()[0][0]

    r1 = c1 / bin_duration_s
    r2 = c2 / bin_duration_s
    r3 = c3 / bin_duration_s

    rate_readings_1[i] = r1
    rate_readings_2[i] = r2
    rate_readings_3[i] = r3

    print(f"Measurement {i+1}: Ch1 = {r1:.4e} cps, Ch2 = {r2:.4e} cps, Ch3 = {r3:.4e} cps")

# -------------------------
# Statistics
# -------------------------
mean1 = np.mean(rate_readings_1)
std1  = np.std(rate_readings_1, ddof=1)

mean2 = np.mean(rate_readings_2)
std2  = np.std(rate_readings_2, ddof=1)

mean3 = np.mean(rate_readings_3)
std3  = np.std(rate_readings_3, ddof=1)

# Apply detection efficiency to channel 1
mean1_tau = tau * mean1
std1_tau  = tau * std1
diff_tau = mean1_tau - mean2

# -------------------------
# Print results
# -------------------------
print("\n=== Results Summary ===")
print(f"Channel 1: Mean = {mean1:.4e} cps, Std Dev = {std1:.4e} cps")
print(f"Channel 2: Mean = {mean2:.4e} cps, Std Dev = {std2:.4e} cps")
print(f"Channel 3: Mean = {mean3:.4e} cps, Std Dev = {std3:.4e} cps")

print("\n=== Channel 1 after detection efficiency correction ===")
print(f"tau * Mean1 = {mean1_tau:.4e} cps")
print(f"tau * Std1  = {std1_tau:.4e} cps")

print(f"(tau * Mean1) - Mean2 = {diff_tau:.4e} cps")